# Imports

In [ ]:
import sys
import os
import mne
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath('../src'))
from A_TriggerChecks.load_trigger_map import load_trigger_map
from C_EEG_Epoching.load_clean_eeg import load_clean_eeg
from C_EEG_Epoching.group_eeg_markers_task1 import group_eeg_markers_task1
from C_EEG_Epoching.group_eeg_markers_task2 import group_eeg_markers_task2
from C_EEG_Epoching.plot_epoch_marker_check import plot_epoch_marker_check
from C_EEG_Epoching.remove_late_click_trials import remove_late_click_trials
from C_EEG_Epoching.save_eeg_epochs import save_eeg_epochs

# Variables

In [ ]:
subjectID = 'Pilote001'
repaired_markers = True
date      = '2026-07-09'
task      = 'Task1_PPS'
data_path =  '/Users/floremunier/Library/CloudStorage/Dropbox-NeuroRestore/Flore Munier-Jolain/PercepPD/Data'
file_name = f"{subjectID}_{date}_{task}"

baseline_time = -0.5
end_time = 2.7

### Load Data

In [ ]:
# Load clean EEG data
raw_cleaned, sfreq, descriptions = load_clean_eeg(data_path, subjectID, date, task)

# Get the path toward the marker .csv file 
if repaired_markers:
    marker_file_name = f'{subjectID}_{date}_{task}_repaired_vmrk_markers.csv'
else:
    marker_file_name = f'{subjectID}_{date}_{task}_vmrk_markers.csv'
# Create a Dataframe from this .csv to work woth these Data
marker_path = os.path.join(data_path, subjectID, task, 'Output', 'Triggers', marker_file_name)
markers_df = pd.read_csv(marker_path)
print(f"[load_vmrk_markers] Loading cleaned markers file:\n  → {marker_path}")

# Extrat trigger value from file
xlsx_path = os.path.join(os.path.dirname(data_path),'Code/EEG_analysis/Scripts/src/Trigger_Values.xlsx')
trigger_map, _ = load_trigger_map(xlsx_path, sheet=task)
print(f"[load_markers_dict] Loading trigger values dictionary:\n  → {xlsx_path}")

### Group events per trials

In [ ]:
max_click_delay = None

if task == 'Task1_PPS':
    events, event_id, events_detailed, removed_markers_df = group_eeg_markers_task1(markers_df,trigger_map)
    max_click_delay = 3

elif task == 'Task2_HitOrMIss':
    events, event_id, events_detailed, removed_markers_df = group_eeg_markers_task2(markers_df,trigger_map)
    max_click_delay = 4

if max_click_delay is not None:
    # Remove the trials where the click arrives after 2.5 sec after the stimuli onset
    events, events_detailed, removed_late_events = remove_late_click_trials(
        events,
        events_detailed,
        sfreq=raw_cleaned.info["sfreq"],
        max_click_delay=max_click_delay
    )

### Create MNE Epochs

In [ ]:
# remove unused event IDs
event_id_clean = {
    name: code
    for name, code in event_id.items()
    if code in np.unique(events[:,2])
}

epochs = mne.Epochs(
    raw_cleaned,
    events,
    event_id=event_id_clean,
    tmin=baseline_time,
    tmax=end_time,
    baseline=(None, 0),
    preload=True,
    reject=dict(
        eeg=150e-6  # 150 µV
    ),
    flat=dict(
        eeg=1e-6
    )
)

### Plot epochs for check

In [ ]:
plot_epoch_marker_check(
    epochs,
    events_detailed,
    raw_cleaned,
    task=task,
    picks=[0]
)

### Save raw epochs

In [ ]:
save_paths = save_eeg_epochs(
    file_path=os.path.join(data_path, subjectID, task,'Output','epoched_EEG'),
    file_name=f"{subjectID}_{date}_{task}",
    epochs=epochs,
    event_id=event_id,
    events_detailed=events_detailed
)